# Egemen — Turkish SLM Training

Fine-tunes **SmolLM2-1.7B** on the Turkish corpus using **QLoRA** (4-bit quantisation + LoRA).  
Designed to run on a free **Kaggle T4** (16 GB VRAM) in ~6–10 hours.

### Steps
1. Install deps
2. Load & stream the corpus from GitHub
3. Load SmolLM2-1.7B in 4-bit
4. Attach LoRA adapters
5. Train
6. Save & (optionally) push to Hugging Face Hub

## 0 · GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — make sure you enabled the GPU accelerator in Kaggle!')

## 1 · Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers==4.51.3 \
    trl==0.16.1 \
    peft==0.15.2 \
    bitsandbytes==0.45.5 \
    accelerate==1.6.0 \
    datasets==3.5.0 \
    huggingface_hub

## 2 · Config — Edit These

In [ ]:
# ── model ──────────────────────────────────────────────────────────────
BASE_MODEL    = "HuggingFaceTB/SmolLM2-1.7B"   # base to fine-tune
OUTPUT_DIR    = "./egemen-output"               # local checkpoint dir
HF_REPO       = "smartlizardpy/egemen"          # set to None to skip upload

# ── data ───────────────────────────────────────────────────────────────
# Raw URLs from https://github.com/smartlizardpy/Egemen
DATA_URLS = [
    "https://raw.githubusercontent.com/smartlizardpy/Egemen/main/scraper/data/people.txt",
    # news files are fetched below via the GitHub API
]
GITHUB_NEWS_API = "https://api.github.com/repos/smartlizardpy/Egemen/git/trees/main?recursive=1"

# ── training ───────────────────────────────────────────────────────────
MAX_SEQ_LEN   = 2048
BATCH_SIZE    = 2          # per device — increase if you have more VRAM
GRAD_ACCUM    = 8          # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 2
WARMUP_RATIO  = 0.03
SAVE_STEPS    = 200

# ── LoRA ───────────────────────────────────────────────────────────────
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

## 3 · Load & Prepare the Dataset

In [ ]:
import requests, io, os
from datasets import Dataset

def fetch_text(url: str) -> str:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.text

def chunk_text(text: str, chunk_size: int = 1500) -> list[str]:
    """Split text into overlapping chunks suitable for CLM training."""
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 80]
    chunks, current = [], []
    current_len = 0
    for para in paragraphs:
        if current_len + len(para) > chunk_size and current:
            chunks.append('\n\n'.join(current))
            current = current[-2:]  # small overlap
            current_len = sum(len(p) for p in current)
        current.append(para)
        current_len += len(para)
    if current:
        chunks.append('\n\n'.join(current))
    return chunks

all_chunks = []

# ── people.txt ─────────────────────────────────────────────────────────
print("Loading people.txt...")
people_text = fetch_text(DATA_URLS[0])
all_chunks.extend(chunk_text(people_text))
print(f"  {len(all_chunks)} chunks so far")

# ── news articles ──────────────────────────────────────────────────────
print("Fetching news file list from GitHub...")
tree = requests.get(GITHUB_NEWS_API, timeout=30).json()
news_files = [
    f"https://raw.githubusercontent.com/smartlizardpy/Egemen/main/{item['path']}"
    for item in tree.get("tree", [])
    if item["path"].startswith("scraper/data/news/") and item["path"].endswith(".txt")
]
print(f"Found {len(news_files)} news files")

for i, url in enumerate(news_files):
    try:
        text = fetch_text(url)
        # strip SOURCE: header
        if text.startswith("SOURCE:"):
            text = text.split("\n\n", 1)[-1]
        all_chunks.extend(chunk_text(text))
    except Exception:
        pass
    if i % 100 == 0:
        print(f"  {i}/{len(news_files)} files, {len(all_chunks)} chunks total")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Sample:\n{all_chunks[0][:300]}")

In [ ]:
import random
random.shuffle(all_chunks)

dataset = Dataset.from_dict({"text": all_chunks})
dataset = dataset.train_test_split(test_size=0.02, seed=42)

print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

## 4 · Load Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
)
model.config.use_cache = False

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 5 · Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6 · Train

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=True,   # pack short samples together — much faster
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

## 7 · Save the Model

In [ ]:
# Save LoRA adapter weights
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# Optional: merge LoRA weights back into the base model (larger file, no peft dependency needed)
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR + "-merged")
tokenizer.save_pretrained(OUTPUT_DIR + "-merged")
print("Merged model saved!")

## 8 · Push to Hugging Face Hub (Optional)

In [ ]:
if HF_REPO:
    from huggingface_hub import login
    # Paste your HF write token here, or set env var HUGGINGFACE_TOKEN
    login(token=os.environ.get("HUGGINGFACE_TOKEN", ""))

    merged.push_to_hub(HF_REPO, private=False)
    tokenizer.push_to_hub(HF_REPO, private=False)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 9 · Quick Test

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged,
    tokenizer=tokenizer,
    device_map="auto",
)

prompts = [
    "Türkiye'nin başkenti",
    "Atatürk, Türk milletinin",
    "İstanbul Boğazı'nın önemi",
]

for prompt in prompts:
    out = pipe(prompt, max_new_tokens=80, temperature=0.7, do_sample=True, repetition_penalty=1.1)
    print(f"Prompt: {prompt}")
    print(f"Output: {out[0]['generated_text']}")
    print("-" * 60)